## imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# regressors
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, HuberRegressor, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

## 1. data audit

In [ ]:
# search kaggle path dynamically
data_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv") and "coaster" in file.lower():
            data_path = os.path.join(root, file)
            break

if data_path is None:
    data_path = "coaster_db.csv"

df = pd.read_csv(data_path)
print(f"loaded dataset from: {data_path}")

# standardizing column names
col_map = {col.lower(): col for col in df.columns}
speed_col = col_map.get('speed_mph', 'speed_mph')
height_col = col_map.get('height_ft', 'height_ft')
length_col = col_map.get('length_ft', 'Length_ft' if 'Length_ft' in df.columns else 'length_ft')
inversions_col = col_map.get('inversions_clean', 'Inversions_clean' if 'Inversions_clean' in df.columns else 'inversions_clean')
gforce_col = col_map.get('gforce_clean', 'Gforce_clean' if 'Gforce_clean' in df.columns else 'gforce_clean')
type_col = col_map.get('type_main', 'Type_Main' if 'Type_Main' in df.columns else 'type_main')

print("\n--- shape ---")
print(df.shape)

print("\n--- dtypes ---")
print(df.dtypes)

print("\n--- missing values ---")
print(df.isnull().sum())

print("\n--- duplicates ---")
print(df.duplicated().sum())

print(f"\n--- target stats ({speed_col}) ---")
print(df[speed_col].describe())

## 2. EDA

In [ ]:
# target distribution
plt.figure(figsize=(8, 4))
sns.histplot(df[speed_col].dropna(), kde=True, color="teal")
plt.title(f"target distribution: {speed_col}")
plt.xlabel("speed (mph)")
plt.show()

# numerical distributions
num_cols_eda = [c for c in [speed_col, height_col, length_col, inversions_col, gforce_col] if c in df.columns]
df[num_cols_eda].hist(figsize=(12, 8), bins=20, color="skyblue", edgecolor="black")
plt.suptitle("numerical features", fontsize=14)
plt.tight_layout()
plt.show()

# correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols_eda].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("correlation matrix")
plt.show()

# scatter plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if height_col in df.columns:
    sns.scatterplot(data=df, x=height_col, y=speed_col, ax=axes[0], color="purple")
    axes[0].set_title(f"{speed_col} vs {height_col}")
if length_col in df.columns:
    sns.scatterplot(data=df, x=length_col, y=speed_col, ax=axes[1], color="orange")
    axes[1].set_title(f"{speed_col} vs {length_col}")
plt.tight_layout()
plt.show()

Target speed looks mostly normal (~48 mph average). Height and length show clear positive relationship with speed, but lots of missing entries will need imputation.

## 3. preprocessing

In [ ]:
# drop rows without target
df_clean = df.dropna(subset=[speed_col]).copy()

# drop duplicate coasters
dup_cols = [c for c in ["Coaster_Name", "Location"] if c in df_clean.columns]
if dup_cols:
    df_clean = df_clean.drop_duplicates(subset=dup_cols).reset_index(drop=True)

# steepness feature
if height_col in df_clean.columns and length_col in df_clean.columns:
    df_clean["height_to_length_ratio"] = df_clean[height_col] / (df_clean[length_col] + 1)
    ratio_col = "height_to_length_ratio"
else:
    ratio_col = None

num_features = [c for c in [height_col, length_col, inversions_col, ratio_col] if c and c in df_clean.columns]
cat_features = [c for c in [type_col] if c in df_clean.columns]

X = df_clean[num_features + cat_features]
y = df_clean[speed_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

def get_preprocessor():
    transformers = []
    if num_features:
        num_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
        transformers.append(("num", num_transformer, num_features))
    if cat_features:
        cat_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])
        transformers.append(("cat", cat_transformer, cat_features))
    return ColumnTransformer(transformers=transformers)

print(f"train shape: {X_train.shape}, test shape: {X_test.shape}")

## 4. regression

In [ ]:
regressors = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(random_state=SEED),
    "Lasso": Lasso(random_state=SEED),
    "ElasticNet": ElasticNet(random_state=SEED),
    "Huber": HuberRegressor(),
    "Bayesian Ridge": BayesianRidge(),
    "KNN Regressor": KNeighborsRegressor(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=SEED),
    "Random Forest Regressor": RandomForestRegressor(random_state=SEED),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=SEED)
}

results = []
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

for name, reg in regressors.items():
    pipe = Pipeline(steps=[("prep", get_preprocessor()), ("reg", reg)])
    
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=kf, scoring="r2")
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    
    results.append({
        "Model": name,
        "Mean CV R2": cv_scores.mean(),
        "Test R2": r2_score(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "MAE": mean_absolute_error(y_test, preds)
    })

results_df = pd.DataFrame(results).sort_values(by="Test R2", ascending=False)
print(results_df.to_string(index=False))

In [ ]:
# hyperparameter tuning
rf_pipe = Pipeline(steps=[("prep", get_preprocessor()), ("reg", RandomForestRegressor(random_state=SEED))])

param_grid = {
    "reg__n_estimators": [50, 100, 200],
    "reg__max_depth": [None, 10, 20],
    "reg__min_samples_split": [2, 5]
}

grid_rf = GridSearchCV(rf_pipe, param_grid, cv=kf, scoring="r2", n_jobs=-1)
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_
tuned_preds = best_rf.predict(X_test)

print("best params:", grid_rf.best_params_)
print(f"tuned test r2: {r2_score(y_test, tuned_preds):.4f}")
print(f"tuned test rmse: {np.sqrt(mean_squared_error(y_test, tuned_preds)):.4f}")

In [ ]:
# feature importances & diagnostic plots
all_feature_names = num_features.copy()
if cat_features and "cat" in best_rf.named_steps["prep"].named_transformers_:
    cat_encoder = best_rf.named_steps["prep"].named_transformers_["cat"].named_steps["encoder"]
    encoded_cat_cols = list(cat_encoder.get_feature_names_out(cat_features))
    all_feature_names.extend(encoded_cat_cols)

importances = best_rf.named_steps["reg"].feature_importances_
feat_imp_df = pd.DataFrame({"Feature": all_feature_names, "Importance": importances}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=feat_imp_df.head(10), x="Importance", y="Feature", palette="mako")
plt.title("feature importances")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_test, tuned_preds, alpha=0.7, color="royalblue")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
axes[0].set_xlabel("actual speed")
axes[0].set_ylabel("predicted speed")
axes[0].set_title("actual vs predicted")

residuals = y_test - tuned_preds
axes[1].scatter(tuned_preds, residuals, alpha=0.7, color="crimson")
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_xlabel("predicted speed")
axes[1].set_ylabel("residuals")
axes[1].set_title("residual plot")
plt.tight_layout()
plt.show()

## 5. classification

In [ ]:
df_clean["speed_high"] = (df_clean[speed_col] > 50).astype(int)
y_cls = df_clean["speed_high"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cls, test_size=0.2, random_state=SEED, stratify=y_cls
)

classifiers = {
    "Logistic Regression": LogisticRegression(random_state=SEED),
    "KNN Classifier": KNeighborsClassifier(),
    "Decision Tree Classifier": DecisionTreeClassifier(random_state=SEED),
    "Naive Bayes": GaussianNB(),
    "SVC": SVC(random_state=SEED),
    "Random Forest Classifier": RandomForestClassifier(random_state=SEED)
}

cls_results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for name, clf in classifiers.items():
    pipe_cls = Pipeline(steps=[("prep", get_preprocessor()), ("clf", clf)])
    
    cv_acc = cross_val_score(pipe_cls, X_train_c, y_train_c, cv=skf, scoring="accuracy")
    pipe_cls.fit(X_train_c, y_train_c)
    preds_c = pipe_cls.predict(X_test_c)
    
    cls_results.append({
        "Model": name,
        "Mean CV Accuracy": cv_acc.mean(),
        "Test Accuracy": accuracy_score(y_test_c, preds_c),
        "Precision": precision_score(y_test_c, preds_c, zero_division=0),
        "Recall": recall_score(y_test_c, preds_c, zero_division=0),
        "F1-Score": f1_score(y_test_c, preds_c, zero_division=0)
    })

cls_df = pd.DataFrame(cls_results).sort_values(by="F1-Score", ascending=False)
print(cls_df.to_string(index=False))

# best classifier evaluation
best_clf_pipe = Pipeline(steps=[("prep", get_preprocessor()), ("clf", RandomForestClassifier(random_state=SEED))])
best_clf_pipe.fit(X_train_c, y_train_c)
best_cls_preds = best_clf_pipe.predict(X_test_c)

print("\n--- classification report (random forest) ---")
print(classification_report(y_test_c, best_cls_preds))

cm = confusion_matrix(y_test_c, best_cls_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Low/Med", "High"], yticklabels=["Low/Med", "High"])
plt.title("confusion matrix - random forest")
plt.xlabel("predicted")
plt.ylabel("actual")
plt.show()